# Mask R-CNN for In-Hand Object Segmentation on Hot3D Dataset

This notebook implements a workflow for using Mask R-CNN to perform 2D segmentation of objects interacting with hands in the Hot3D dataset. The implementation:

1. Focuses exclusively on segmentation masks (no bounding boxes)
2. Uses a robust definition of "in-hand objects" based on spatial proximity and object velocity
3. Evaluates performance using mean Intersection over Union (mIoU)
4. Uses TensorBoard for monitoring training progress
5. Includes functions to process and cache sequences to avoid reloading
6. Provides inference capabilities on new images


## 1. Setup and Configuration

In [ ]:
import os
import re
import sys
import glob
import random
import pickle
import traceback
import numpy as np
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.mask_rcnn import MaskRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as F
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm
import time
import json
from torch.utils.data import Dataset, DataLoader
import cv2
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter

# Configure project directory structure
home = os.path.expanduser("/storage/aspoto")
hot3d_dataset_path = os.path.join(home, "hot3d/hot3d/dataset")
project_dir = os.path.join(home, "hot3d/hot3d/mask_rcnn_project")

# Create project directories
models_dir = os.path.join(project_dir, "models")
log_dir = os.path.join(project_dir, "logs")
results_dir = os.path.join(project_dir, "results")
visualizations_dir = os.path.join(project_dir, "visualizations")
dataset_cache_dir = os.path.join(project_dir, "dataset_cache")
tensorboard_dir = os.path.join(project_dir, "tensorboard_logs")

for dir_path in [project_dir, models_dir, results_dir, visualizations_dir, log_dir, dataset_cache_dir, tensorboard_dir]:
    os.makedirs(dir_path, exist_ok=True)

# Add Hot3D to the path
sys.path.append('/storage/aspoto/hot3d/hot3d')

# Import Hot3D libraries
from dataset_api import Hot3dDataProvider
from data_loaders.loader_object_library import load_object_library
from projectaria_tools.core.sensor_data import TimeDomain, TimeQueryOptions
from projectaria_tools.core.stream_id import StreamId
from data_loaders.loader_hand_poses import LEFT_HAND_INDEX, RIGHT_HAND_INDEX

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Check available device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define visualization colors for masks
BLUE_MASK_COLOR = [0, 110, 255]  # Blue for masks
MASK_ALPHA = 0.6  # Mask opacity

## 2. Simplified Dataset Class for In-Hand Objects

Creating a simple dataset class that focuses only on in-hand objects:

In [ ]:
class SimpleInHandObjectDataset(Dataset):
    """
    A simplified dataset for in-hand object segmentation that avoids pickle compatibility issues
    """
    def __init__(self, samples=None):
        """
        Args:
            samples: List of dictionaries containing images and targets
        """
        self.samples = samples if samples is not None else []
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = sample['image']
        targets = sample['targets']
        
        # Make sure image is a tensor
        if not isinstance(image, torch.Tensor):
            image = F.to_tensor(image)
        
        # Make sure masks, boxes, and labels exist and are properly formatted
        if 'masks' in targets and len(targets['masks']) > 0:
            # Ensure boxes exist
            if 'boxes' not in targets or len(targets['boxes']) == 0:
                boxes = []
                for mask in targets['masks']:
                    # Get bounding box from mask
                    if isinstance(mask, torch.Tensor):
                        mask_np = mask.cpu().numpy()
                    else:
                        mask_np = mask
                        
                    # Find non-zero elements (mask pixels)
                    rows = np.any(mask_np, axis=1)
                    cols = np.any(mask_np, axis=0)
                    ymin, ymax = np.where(rows)[0][[0, -1]] if np.any(rows) else (0, 0)
                    xmin, xmax = np.where(cols)[0][[0, -1]] if np.any(cols) else (0, 0)
                    
                    # Create box [x1, y1, x2, y2]
                    box = [float(xmin), float(ymin), float(xmax), float(ymax)]
                    boxes.append(box)
                
                targets['boxes'] = torch.tensor(boxes, dtype=torch.float32)
            
            # Ensure labels exist and are all set to 1 (in-hand object)
            if 'labels' not in targets or len(targets['labels']) == 0:
                targets['labels'] = torch.ones(len(targets['masks']), dtype=torch.int64)
            else:
                # Force all labels to be 1 (in-hand object)
                targets['labels'] = torch.ones_like(targets['labels'])
                
            # Make sure masks are tensors
            if not isinstance(targets['masks'], torch.Tensor):
                targets['masks'] = torch.tensor(targets['masks'], dtype=torch.uint8)
        else:
            # No masks - create empty tensors
            h, w = image.shape[-2], image.shape[-1]
            targets['boxes'] = torch.zeros((0, 4), dtype=torch.float32)
            targets['labels'] = torch.zeros(0, dtype=torch.int64)
            targets['masks'] = torch.zeros((0, h, w), dtype=torch.uint8)
        
        return image, targets
    
    def save_to_file(self, file_path):
        """Save dataset to a simple dictionary format"""
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        
        # Convert to a simpler format for saving
        simple_samples = []
        
        for sample in self.samples:
            # Convert image to numpy if it's a tensor
            if isinstance(sample['image'], torch.Tensor):
                image = sample['image'].cpu().numpy()
            else:
                image = sample['image']
                
            # Convert targets to numpy
            targets = {}
            for k, v in sample['targets'].items():
                if isinstance(v, torch.Tensor):
                    targets[k] = v.cpu().numpy()
                else:
                    targets[k] = v
            
            # Store as a simple dict
            simple_samples.append({
                'image': image,
                'targets': targets,
                'metadata': sample.get('metadata', {})
            })
        
        # Save using numpy's save function (more reliable than pickle)
        np.save(file_path, simple_samples)
        print(f"Dataset saved to {file_path} with {len(simple_samples)} samples")
    
    @classmethod
    def load_from_file(cls, file_path):
        """Load dataset from a file"""
        if not os.path.exists(file_path):
            print(f"Warning: File {file_path} not found")
            return cls([])
        
        try:
            # Load the numpy array
            simple_samples = np.load(file_path, allow_pickle=True)
            
            # Create a new dataset
            dataset = cls(simple_samples)
            print(f"Dataset loaded from {file_path} with {len(simple_samples)} samples")
            return dataset
            
        except Exception as e:
            print(f"Error loading dataset from {file_path}: {e}")
            traceback.print_exc()
            return cls([])

## 3. Function to Process Hot3D Sequences and Create Dataset

In [ ]:
def process_hot3d_sequences(sequence_paths, num_samples_per_sequence=20, hand_proximity_threshold=0.01, object_velocity_threshold=0.01):
    """
    Process Hot3D sequences to extract in-hand object data
    
    Args:
        sequence_paths: List of paths to Hot3D sequences
        num_samples_per_sequence: Number of frames to sample from each sequence
        hand_proximity_threshold: Distance threshold in meters to consider an object "in-hand" (1cm)
        object_velocity_threshold: Velocity threshold in m/s to consider an object being manipulated (1cm/s)
        
    Returns:
        Dataset containing the processed samples
    """
    # Path to object library
    object_library_path = os.path.join(hot3d_dataset_path, "assets")
    
    # Load object library
    print("Loading object library...")
    object_library = load_object_library(object_library_folderpath=object_library_path)
    
    # List to store all samples
    all_samples = []
    
    # Test set participant IDs
    TEST_SET_PARTICIPANT_ID = ["P0004", "P0005", "P0006", "P0008", "P0016", "P0020"]
    
    # Process each sequence
    for sequence_path in tqdm(sequence_paths, desc="Processing sequences"):
        try:
            # Extract participant ID
            sequence_name = os.path.basename(sequence_path)
            match = re.match(r'(P\d+)_', sequence_name)
            participant_id = match.group(1) if match else None
            
            print(f"Processing sequence: {sequence_name}, Participant: {participant_id}")
            
            # Initialize data provider
            hot3d_data_provider = Hot3dDataProvider(
                sequence_folder=sequence_path,
                object_library=object_library
            )
            
            # Check if this is a test sequence
            is_test = participant_id in TEST_SET_PARTICIPANT_ID
            print(f"  Is test sequence: {is_test}")
            
            # Get providers
            device_data_provider = hot3d_data_provider.device_data_provider
            object_mask_provider = hot3d_data_provider.object_mask_data_provider
            object_pose_provider = hot3d_data_provider.object_pose_data_provider
            hand_pose_provider = hot3d_data_provider.hand_pose_data_provider
            
            # Get timestamps
            timestamps = device_data_provider.get_sequence_timestamps()
            
            # Get image stream IDs
            image_stream_ids = device_data_provider.get_image_stream_ids()
            
            # Select RGB stream if available, otherwise use SLAM-LEFT
            if StreamId("214-1") in image_stream_ids:
                stream_id = StreamId("214-1")  # RGB
            else:
                stream_id = StreamId("1201-1")  # SLAM-LEFT
            
            # Sample timestamps
            sampled_indices = np.linspace(
                0, len(timestamps)-1, 
                min(num_samples_per_sequence, len(timestamps)), 
                dtype=int
            )
            
            # Process each sampled timestamp
            for idx in sampled_indices:
                timestamp_ns = timestamps[idx]
                
                # Get image data
                try:
                    image_data = device_data_provider.get_image(timestamp_ns, stream_id)
                except Exception as e:
                    print(f"Error getting image at {timestamp_ns}: {e}")
                    continue
                
                # Initialize targets dictionary
                targets = {'masks': []}
                
                # Determine which objects are "in-hand" (for training data only)
                in_hand_object_uids = set()
                
                if not is_test and object_pose_provider and hand_pose_provider:
                    # Get object poses
                    object_poses = object_pose_provider.get_pose_at_timestamp(
                        timestamp_ns=timestamp_ns,
                        time_query_options=TimeQueryOptions.CLOSEST,
                        time_domain=TimeDomain.TIME_CODE,
                    )
                    
                    # Get hand poses
                    hand_poses = hand_pose_provider.get_pose_at_timestamp(
                        timestamp_ns=timestamp_ns,
                        time_query_options=TimeQueryOptions.CLOSEST,
                        time_domain=TimeDomain.TIME_CODE,
                    )
                    
                    # Skip if no poses are available
                    if object_poses and hand_poses:
                        # Get previous timestamp for velocity calculation
                        prev_idx = max(0, idx - 5)  # Use a few frames back for velocity
                        prev_timestamp_ns = timestamps[prev_idx]
                        
                        # Get previous object poses
                        prev_object_poses = object_pose_provider.get_pose_at_timestamp(
                            timestamp_ns=prev_timestamp_ns,
                            time_query_options=TimeQueryOptions.CLOSEST,
                            time_domain=TimeDomain.TIME_CODE,
                        )
                        
                        # Calculate time difference in seconds
                        time_diff = (timestamp_ns - prev_timestamp_ns) / 1e9
                        
                        # Check each object
                        for obj_uid in object_poses.pose_dict.keys():
                            # Skip if this object doesn't have a pose
                            if obj_uid not in object_poses.pose_dict:
                                continue
                            
                            # Get object pose
                            obj_pose = object_poses.pose_dict[obj_uid]
                            
                            # Check hand proximity
                            is_near_hand = False
                            
                            # Get object position
                            obj_position = np.array(obj_pose.transform[:3, 3])
                            
                            # Check distance to each hand
                            for hand_id in [LEFT_HAND_INDEX, RIGHT_HAND_INDEX]:
                                if hand_id in hand_poses.pose_dict:
                                    hand_pose = hand_poses.pose_dict[hand_id]
                                    hand_position = np.array(hand_pose.transform[:3, 3])
                                    
                                    # Calculate distance
                                    distance = np.linalg.norm(obj_position - hand_position)
                                    
                                    # Check if within threshold
                                    if distance < hand_proximity_threshold:
                                        is_near_hand = True
                                        break
                            
                            # Calculate object velocity
                            is_moving = False
                            if prev_object_poses and obj_uid in prev_object_poses.pose_dict and time_diff > 0:
                                prev_obj_pose = prev_object_poses.pose_dict[obj_uid]
                                prev_obj_position = np.array(prev_obj_pose.transform[:3, 3])
                                
                                # Calculate displacement
                                displacement = np.linalg.norm(obj_position - prev_obj_position)
                                
                                # Calculate velocity
                                velocity = displacement / time_diff
                                
                                # Check if velocity exceeds threshold
                                is_moving = velocity > object_velocity_threshold
                            
                            # If both criteria are met, consider this an "in-hand" object
                            if is_near_hand and is_moving:
                                in_hand_object_uids.add(obj_uid)
                
                # Get object masks
                if object_mask_provider:
                    try:
                        mask_collection = object_mask_provider.get_mask_at_timestamp(
                            stream_id=stream_id,
                            timestamp_ns=timestamp_ns,
                            time_query_options=TimeQueryOptions.CLOSEST,
                            time_domain=TimeDomain.TIME_CODE,
                        )
                        
                        if mask_collection:
                            # For test set or if in-hand objects couldn't be determined, use all available object masks
                            objects_to_use = mask_collection.mask_collection.object_uid_list \
                                if (is_test or not in_hand_object_uids) else in_hand_object_uids
                            
                            # Extract object masks
                            for obj_uid in objects_to_use:
                                if obj_uid in mask_collection.mask_collection.masks:
                                    mask_data = mask_collection.mask_collection.masks[obj_uid]
                                    
                                    if mask_data is not None:
                                        # Add mask to targets
                                        targets['masks'].append(mask_data)
                    except Exception as e:
                        print(f"Error getting object masks: {e}")
                
                # Skip frames without objects for training data
                if not is_test and len(targets['masks']) == 0:
                    continue
                
                # Add sample
                all_samples.append({
                    'image': image_data,
                    'targets': targets,
                    'metadata': {
                        'sequence': sequence_name,
                        'timestamp': timestamp_ns,
                        'is_test': is_test
                    }
                })
                
        except Exception as e:
            print(f"Error processing sequence {sequence_path}: {e}")
            traceback.print_exc()
    
    # Create dataset
    dataset = SimpleInHandObjectDataset(all_samples)
    print(f"Created dataset with {len(dataset)} samples")
    
    return dataset

## 4. Find Hot3D Sequences and Split into Train/Val/Test Sets

In [ ]:
def find_sequences(dataset_path):
    """Find all Hot3D sequences in the dataset path"""
    sequences = []
    for item in os.listdir(dataset_path):
        item_path = os.path.join(dataset_path, item)
        # Skip the assets folder and non-directories
        if item == "assets" or not os.path.isdir(item_path):
            continue
        # Check if this is a valid Hot3D sequence (has recording.vrs file)
        vrs_path = os.path.join(item_path, "recording.vrs")
        if os.path.exists(vrs_path):
            sequences.append(item_path)
    
    return sequences

def split_sequences(sequences):
    """
    Split sequences according to the official HOT3D criteria where
    TEST_SET_PARTICIPANT_ID = ["P0004", "P0005", "P0006", "P0008", "P0016", "P0020"]
    """
    TEST_SET_PARTICIPANT_ID = ["P0004", "P0005", "P0006", "P0008", "P0016", "P0020"]
    
    train_sequences = []
    test_sequences = []
    
    # Helper function to extract participant ID
    def get_participant_id(sequence_path):
        # Extract sequence name from path 
        sequence_name = os.path.basename(sequence_path)
        # Extract participant ID using regex
        match = re.match(r'(P\d+)_', sequence_name)
        if match:
            return match.group(1)
        return None
    
    for sequence in sequences:
        participant_id = get_participant_id(sequence)
        if participant_id in TEST_SET_PARTICIPANT_ID:
            test_sequences.append(sequence)
        else:
            train_sequences.append(sequence)
    
    return train_sequences, test_sequences

# Find sequences and split them
print("Finding and splitting Hot3D sequences...")
all_sequences = find_sequences(hot3d_dataset_path)
train_sequences, test_sequences = split_sequences(all_sequences)

# Split train into train/val
train_sequences, val_sequences = train_test_split(train_sequences, test_size=0.2, random_state=42)

print(f"Found {len(all_sequences)} total sequences")
print(f"Train: {len(train_sequences)} sequences")
print(f"Validation: {len(val_sequences)} sequences")
print(f"Test: {len(test_sequences)} sequences")

In [ ]:
# Define cache file paths
train_cache_path = os.path.join(dataset_cache_dir, "train_dataset.npy")
val_cache_path = os.path.join(dataset_cache_dir, "val_dataset.npy")
test_cache_path = os.path.join(dataset_cache_dir, "test_dataset.npy")

# Check if cached datasets exist - otherwise, process and save
if os.path.exists(train_cache_path) and os.path.exists(val_cache_path) and os.path.exists(test_cache_path):
    print("Loading cached datasets...")
    train_dataset = SimpleInHandObjectDataset.load_from_file(train_cache_path)
    val_dataset = SimpleInHandObjectDataset.load_from_file(val_cache_path)
    test_dataset = SimpleInHandObjectDataset.load_from_file(test_cache_path)
else:
    print("Processing sequences and creating datasets...")
    
    # Process train sequences
    print("\nProcessing training sequences...")
    train_dataset = process_hot3d_sequences(
        train_sequences, 
        num_samples_per_sequence=20,
        hand_proximity_threshold=0.01,  # 1cm
        object_velocity_threshold=0.01  # 1cm/s
    )
    train_dataset.save_to_file(train_cache_path)
    
    # Process validation sequences
    print("\nProcessing validation sequences...")
    val_dataset = process_hot3d_sequences(
        val_sequences, 
        num_samples_per_sequence=20,
        hand_proximity_threshold=0.01,
        object_velocity_threshold=0.01
    )
    val_dataset.save_to_file(val_cache_path)
    
    # Process test sequences
    print("\nProcessing test sequences...")
    test_dataset = process_hot3d_sequences(
        test_sequences, 
        num_samples_per_sequence=20,
        hand_proximity_threshold=0.01,
        object_velocity_threshold=0.01
    )
    test_dataset.save_to_file(test_cache_path)

# Report dataset statistics
print(f"Dataset statistics:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Test samples: {len(test_dataset)}")

In [ ]:
class MaskRCNNModel:
    """
    Wrapper class for Mask R-CNN model with training and evaluation functions
    """
    def __init__(self, num_classes=2, pretrained=True):
        """
        Initialize the Mask R-CNN model
        
        Args:
            num_classes: Number of classes (background + object classes)
            pretrained: Whether to use pretrained weights
        """
        # Initialize model
        if pretrained:
            weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
            self.model = maskrcnn_resnet50_fpn(weights=weights)
        else:
            self.model = maskrcnn_resnet50_fpn()
        
        # Get number of input features
        in_features = self.model.roi_heads.box_predictor.cls_score.in_features
        
        # Replace the box predictor and mask predictor with new ones
        # for our binary classification task (background + in-hand object)
        self.model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
            in_features, num_classes
        )
        
        # Get the number of input features for mask predictor
        in_features_mask = self.model.roi_heads.mask_predictor.conv5_mask.in_channels
        hidden_layer = 256
        
        # Replace the mask predictor
        self.model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
            in_features_mask, hidden_layer, num_classes
        )
        
        # Move model to the appropriate device
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
    
    def train_one_epoch(self, data_loader, optimizer, epoch, print_freq=10):
        """
        Train the model for one epoch
        
        Args:
            data_loader: DataLoader for training data
            optimizer: Optimizer for model parameters
            epoch: Current epoch number
            print_freq: How often to print progress
            
        Returns:
            Average loss for the epoch
        """
        self.model.train()
        total_loss = 0.0
        running_loss = 0.0
        
        with tqdm(data_loader, desc=f"Epoch {epoch}") as pbar:
            for i, (images, targets) in enumerate(pbar):
                # Move data to device
                images = [image.to(self.device) for image in images]
                targets = [{k: v.to(self.device) for k, v in t.items()} for t in targets]
                
                # Zero out gradients
                optimizer.zero_grad()
                
                # Forward pass
                loss_dict = self.model(images, targets)
                
                # Calculate total loss
                losses = sum(loss for loss in loss_dict.values())
                
                # Backward pass and optimize
                losses.backward()
                optimizer.step()
                
                # Update running loss
                running_loss += losses.item()
                total_loss += losses.item()
                
                # Update progress bar
                if i % print_freq == 0:
                    pbar.set_postfix({"loss": running_loss / (i + 1)})
        
        # Calculate average loss
        avg_loss = total_loss / len(data_loader)
        return avg_loss
    
    def evaluate(self, data_loader, iou_threshold=0.5):
        """
        Evaluate the model on validation/test data
        
        Args:
            data_loader: DataLoader for validation/test data
            iou_threshold: IoU threshold for considering a prediction as correct
            
        Returns:
            Dictionary with evaluation metrics
        """
        self.model.eval()
        
        # Lists to store metrics
        mask_ious = []
        precisions = []
        recalls = []
        f1_scores = []
        
        with torch.no_grad():
            for images, targets in tqdm(data_loader, desc="Evaluating"):
                # Move images to device
                images = [image.to(self.device) for image in images]
                
                # Get predictions
                predictions = self.model(images)
                
                # Process each image
                for i, (pred, target) in enumerate(zip(predictions, targets)):
                    # Move target to CPU for evaluation
                    target = {k: v.cpu() for k, v in target.items()}
                    
                    # Get predicted masks
                    pred_masks = pred['masks'].cpu()
                    pred_scores = pred['scores'].cpu()
                    
                    # Get target masks
                    target_masks = target['masks']
                    
                    # Skip images without target masks
                    if len(target_masks) == 0:
                        continue
                    
                    # Convert target masks to binary format (0 or 1)
                    target_masks = target_masks > 0.5
                    
                    # Use only high-confidence predictions
                    high_conf_indices = pred_scores > 0.5
                    if high_conf_indices.sum() == 0:
                        # No high-confidence predictions
                        precisions.append(0.0)
                        recalls.append(0.0)
                        f1_scores.append(0.0)
                        continue
                    
                    pred_masks = pred_masks[high_conf_indices]
                    
                    # Convert predicted masks to binary format
                    pred_masks = pred_masks > 0.5
                    
                    # Skip if no predicted masks after thresholding
                    if len(pred_masks) == 0:
                        precisions.append(0.0)
                        recalls.append(0.0)
                        f1_scores.append(0.0)
                        continue
                    
                    # Calculate IoU for each pair of predicted and target mask
                    ious = []
                    for pred_mask in pred_masks:
                        pred_mask = pred_mask.squeeze()
                        pred_mask_area = pred_mask.sum().item()
                        
                        # If predicted mask is empty, IoU is 0
                        if pred_mask_area == 0:
                            ious.append(0.0)
                            continue
                        
                        mask_ious_per_pred = []
                        for target_mask in target_masks:
                            target_mask = target_mask.squeeze()
                            target_mask_area = target_mask.sum().item()
                            
                            # If target mask is empty, IoU is 0
                            if target_mask_area == 0:
                                mask_ious_per_pred.append(0.0)
                                continue
                            
                            # Calculate intersection and union
                            intersection = (pred_mask & target_mask).sum().item()
                            union = pred_mask_area + target_mask_area - intersection
                            
                            # Calculate IoU
                            iou = intersection / union if union > 0 else 0.0
                            mask_ious_per_pred.append(iou)
                        
                        # Get best IoU for this predicted mask
                        if mask_ious_per_pred:
                            ious.append(max(mask_ious_per_pred))
                        else:
                            ious.append(0.0)
                    
                    # Calculate precision, recall, and F1 score
                    true_positives = sum(1 for iou in ious if iou >= iou_threshold)
                    false_positives = len(pred_masks) - true_positives
                    false_negatives = max(0, len(target_masks) - true_positives)
                    
                    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0.0
                    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
                    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                    
                    # Store metrics
                    if ious:
                        mask_ious.extend(ious)
                    precisions.append(precision)
                    recalls.append(recall)
                    f1_scores.append(f1)
        
        # Calculate average metrics
        avg_mask_iou = sum(mask_ious) / len(mask_ious) if mask_ious else 0.0
        avg_precision = sum(precisions) / len(precisions) if precisions else 0.0
        avg_recall = sum(recalls) / len(recalls) if recalls else 0.0
        avg_f1_score = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
        
        # Return evaluation metrics
        return {
            "mIoU": avg_mask_iou,
            "Precision": avg_precision,
            "Recall": avg_recall,
            "F1": avg_f1_score
        }
    
    def save(self, path):
        """Save model to a file"""
        torch.save(self.model.state_dict(), path)
        print(f"Model saved to {path}")
    
    def load(self, path):
        """Load model from a file"""
        if os.path.exists(path):
            self.model.load_state_dict(torch.load(path, map_location=self.device))
            print(f"Model loaded from {path}")
            return True
        else:
            print(f"Warning: Model file {path} not found")
            return False
    
    def predict(self, image, score_threshold=0.5):
        """
        Make a prediction on a single image
        
        Args:
            image: Input image (PIL Image or tensor)
            score_threshold: Confidence threshold for predictions
            
        Returns:
            Predicted masks and scores
        """
        self.model.eval()
        
        # Convert image to tensor if needed
        if not isinstance(image, torch.Tensor):
            image = F.to_tensor(image)
        
        # Add batch dimension and move to device
        image = image.unsqueeze(0).to(self.device)
        
        # Make prediction
        with torch.no_grad():
            prediction = self.model(image)[0]
        
        # Get masks and scores
        masks = prediction['masks'].cpu()
        scores = prediction['scores'].cpu()
        
        # Apply score threshold
        high_conf_indices = scores >= score_threshold
        masks = masks[high_conf_indices]
        scores = scores[high_conf_indices]
        
        return masks, scores

In [ ]:
# Cell 5: Process Sequences and Save Datasets

# Define cache file paths
train_cache_path = os.path.join(dataset_cache_dir, "train_dataset.npy")
val_cache_path = os.path.join(dataset_cache_dir, "val_dataset.npy")
test_cache_path = os.path.join(dataset_cache_dir, "test_dataset.npy")

# Check if cached datasets exist - otherwise, process and save
if os.path.exists(train_cache_path) and os.path.exists(val_cache_path) and os.path.exists(test_cache_path):
    print("Loading cached datasets...")
    train_dataset = SimpleInHandObjectDataset.load_from_file(train_cache_path)
    val_dataset = SimpleInHandObjectDataset.load_from_file(val_cache_path)
    test_dataset = SimpleInHandObjectDataset.load_from_file(test_cache_path)
else:
    print("Processing sequences and creating datasets...")
    
    # Process train sequences
    print("\nProcessing training sequences...")
    train_dataset = process_hot3d_sequences(
        train_sequences, 
        num_samples_per_sequence=20,
        hand_proximity_threshold=0.01,  # 1cm
        object_velocity_threshold=0.01  # 1cm/s
    )
    train_dataset.save_to_file(train_cache_path)
    
    # Process validation sequences
    print("\nProcessing validation sequences...")
    val_dataset = process_hot3d_sequences(
        val_sequences, 
        num_samples_per_sequence=20,
        hand_proximity_threshold=0.01,
        object_velocity_threshold=0.01
    )
    val_dataset.save_to_file(val_cache_path)
    
    # Process test sequences
    print("\nProcessing test sequences...")
    test_dataset = process_hot3d_sequences(
        test_sequences, 
        num_samples_per_sequence=20,
        hand_proximity_threshold=0.01,
        object_velocity_threshold=0.01
    )
    test_dataset.save_to_file(test_cache_path)

# Report dataset statistics
print(f"Dataset statistics:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Test samples: {len(test_dataset)}")

# Cell 6: Model Definition and Training Utilities

class MaskRCNNModel:
    """
    Wrapper class for Mask R-CNN model with training and evaluation functions
    """
    def __init__(self, num_classes=2, pretrained=True):
        """
        Initialize the Mask R-CNN model
        
        Args:
            num_classes: Number of classes (background + object classes)
            pretrained: Whether to use pretrained weights
        """
        # Initialize model
        if pretrained:
            weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
            self.model = maskrcnn_resnet50_fpn(weights=weights)
        else:
            self.model = maskrcnn_resnet50_fpn()
        
        # Get number of input features
        in_features = self.model.roi_heads.box_predictor.cls_score.in_features
        
        # Replace the box predictor and mask predictor with new ones
        # for our binary classification task (background + in-hand object)
        self.model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
            in_features, num_classes
        )
        
        # Get the number of input features for mask predictor
        in_features_mask = self.model.roi_heads.mask_predictor.conv5_mask.in_channels
        hidden_layer = 256
        
        # Replace the mask predictor
        self.model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
            in_features_mask, hidden_layer, num_classes
        )
        
        # Move model to the appropriate device
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
    
    def train_one_epoch(self, data_loader, optimizer, epoch, print_freq=10):
        """
        Train the model for one epoch
        
        Args:
            data_loader: DataLoader for training data
            optimizer: Optimizer for model parameters
            epoch: Current epoch number
            print_freq: How often to print progress
            
        Returns:
            Average loss for the epoch
        """
        self.model.train()
        total_loss = 0.0
        running_loss = 0.0
        
        with tqdm(data_loader, desc=f"Epoch {epoch}") as pbar:
            for i, (images, targets) in enumerate(pbar):
                # Move data to device
                images = [image.to(self.device) for image in images]
                targets = [{k: v.to(self.device) for k, v in t.items()} for t in targets]
                
                # Zero out gradients
                optimizer.zero_grad()
                
                # Forward pass
                loss_dict = self.model(images, targets)
                
                # Calculate total loss
                losses = sum(loss for loss in loss_dict.values())
                
                # Backward pass and optimize
                losses.backward()
                optimizer.step()
                
                # Update running loss
                running_loss += losses.item()
                total_loss += losses.item()
                
                # Update progress bar
                if i % print_freq == 0:
                    pbar.set_postfix({"loss": running_loss / (i + 1)})
        
        # Calculate average loss
        avg_loss = total_loss / len(data_loader)
        return avg_loss
    
    def evaluate(self, data_loader, iou_threshold=0.5):
        """
        Evaluate the model on validation/test data
        
        Args:
            data_loader: DataLoader for validation/test data
            iou_threshold: IoU threshold for considering a prediction as correct
            
        Returns:
            Dictionary with evaluation metrics
        """
        self.model.eval()
        
        # Lists to store metrics
        mask_ious = []
        precisions = []
        recalls = []
        f1_scores = []
        
        with torch.no_grad():
            for images, targets in tqdm(data_loader, desc="Evaluating"):
                # Move images to device
                images = [image.to(self.device) for image in images]
                
                # Get predictions
                predictions = self.model(images)
                
                # Process each image
                for i, (pred, target) in enumerate(zip(predictions, targets)):
                    # Move target to CPU for evaluation
                    target = {k: v.cpu() for k, v in target.items()}
                    
                    # Get predicted masks
                    pred_masks = pred['masks'].cpu()
                    pred_scores = pred['scores'].cpu()
                    
                    # Get target masks
                    target_masks = target['masks']
                    
                    # Skip images without target masks
                    if len(target_masks) == 0:
                        continue
                    
                    # Convert target masks to binary format (0 or 1)
                    target_masks = target_masks > 0.5
                    
                    # Use only high-confidence predictions
                    high_conf_indices = pred_scores > 0.5
                    if high_conf_indices.sum() == 0:
                        # No high-confidence predictions
                        precisions.append(0.0)
                        recalls.append(0.0)
                        f1_scores.append(0.0)
                        continue
                    
                    pred_masks = pred_masks[high_conf_indices]
                    
                    # Convert predicted masks to binary format
                    pred_masks = pred_masks > 0.5
                    
                    # Skip if no predicted masks after thresholding
                    if len(pred_masks) == 0:
                        precisions.append(0.0)
                        recalls.append(0.0)
                        f1_scores.append(0.0)
                        continue
                    
                    # Calculate IoU for each pair of predicted and target mask
                    ious = []
                    for pred_mask in pred_masks:
                        pred_mask = pred_mask.squeeze()
                        pred_mask_area = pred_mask.sum().item()
                        
                        # If predicted mask is empty, IoU is 0
                        if pred_mask_area == 0:
                            ious.append(0.0)
                            continue
                        
                        mask_ious_per_pred = []
                        for target_mask in target_masks:
                            target_mask = target_mask.squeeze()
                            target_mask_area = target_mask.sum().item()
                            
                            # If target mask is empty, IoU is 0
                            if target_mask_area == 0:
                                mask_ious_per_pred.append(0.0)
                                continue
                            
                            # Calculate intersection and union
                            intersection = (pred_mask & target_mask).sum().item()
                            union = pred_mask_area + target_mask_area - intersection
                            
                            # Calculate IoU
                            iou = intersection / union if union > 0 else 0.0
                            mask_ious_per_pred.append(iou)
                        
                        # Get best IoU for this predicted mask
                        if mask_ious_per_pred:
                            ious.append(max(mask_ious_per_pred))
                        else:
                            ious.append(0.0)
                    
                    # Calculate precision, recall, and F1 score
                    true_positives = sum(1 for iou in ious if iou >= iou_threshold)
                    false_positives = len(pred_masks) - true_positives
                    false_negatives = max(0, len(target_masks) - true_positives)
                    
                    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0.0
                    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
                    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                    
                    # Store metrics
                    if ious:
                        mask_ious.extend(ious)
                    precisions.append(precision)
                    recalls.append(recall)
                    f1_scores.append(f1)
        
        # Calculate average metrics
        avg_mask_iou = sum(mask_ious) / len(mask_ious) if mask_ious else 0.0
        avg_precision = sum(precisions) / len(precisions) if precisions else 0.0
        avg_recall = sum(recalls) / len(recalls) if recalls else 0.0
        avg_f1_score = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
        
        # Return evaluation metrics
        return {
            "mIoU": avg_mask_iou,
            "Precision": avg_precision,
            "Recall": avg_recall,
            "F1": avg_f1_score
        }
    
    def save(self, path):
        """Save model to a file"""
        torch.save(self.model.state_dict(), path)
        print(f"Model saved to {path}")
    
    def load(self, path):
        """Load model from a file"""
        if os.path.exists(path):
            self.model.load_state_dict(torch.load(path, map_location=self.device))
            print(f"Model loaded from {path}")
            return True
        else:
            print(f"Warning: Model file {path} not found")
            return False
    
    def predict(self, image, score_threshold=0.5):
        """
        Make a prediction on a single image
        
        Args:
            image: Input image (PIL Image or tensor)
            score_threshold: Confidence threshold for predictions
            
        Returns:
            Predicted masks and scores
        """
        self.model.eval()
        
        # Convert image to tensor if needed
        if not isinstance(image, torch.Tensor):
            image = F.to_tensor(image)
        
        # Add batch dimension and move to device
        image = image.unsqueeze(0).to(self.device)
        
        # Make prediction
        with torch.no_grad():
            prediction = self.model(image)[0]
        
        # Get masks and scores
        masks = prediction['masks'].cpu()
        scores = prediction['scores'].cpu()
        
        # Apply score threshold
        high_conf_indices = scores >= score_threshold
        masks = masks[high_conf_indices]
        scores = scores[high_conf_indices]
        
        return masks, scores

# Cell 7: Training and Evaluation Functions

def collate_fn(batch):
    """
    Custom collate function for data loader
    """
    return tuple(zip(*batch))

def create_data_loaders(train_dataset, val_dataset, test_dataset, batch_size=4):
    """
    Create data loaders for training, validation, and testing
    """
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=4
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=4
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=4
    )
    
    return train_loader, val_loader, test_loader

def train_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.001, model_save_path=None):
    """
    Train the model
    
    Args:
        model: MaskRCNNModel instance
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        num_epochs: Number of epochs to train
        learning_rate: Learning rate for optimizer
        model_save_path: Path to save the best model
        
    Returns:
        Training history dictionary
    """
    # Initialize optimizer
    params = [p for p in model.model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=learning_rate, momentum=0.9, weight_decay=0.0005)
    
    # Learning rate scheduler
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    
    # TensorBoard writer
    writer = SummaryWriter(tensorboard_dir)
    
    # Training history
    history = {
        'train_loss': [],
        'val_miou': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': []
    }
    
    # Best validation mIoU
    best_miou = 0.0
    
    # Train for specified number of epochs
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        # Train one epoch
        train_loss = model.train_one_epoch(train_loader, optimizer, epoch+1)
        
        # Update learning rate
        lr_scheduler.step()
        
        # Evaluate on validation set
        val_metrics = model.evaluate(val_loader)
        
        # Print metrics
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Validation Metrics:")
        print(f"  mIoU: {val_metrics['mIoU']:.4f}")
        print(f"  Precision: {val_metrics['Precision']:.4f}")
        print(f"  Recall: {val_metrics['Recall']:.4f}")
        print(f"  F1 Score: {val_metrics['F1']:.4f}")
        
        # Update history
        history['train_loss'].append(train_loss)
        history['val_miou'].append(val_metrics['mIoU'])
        history['val_precision'].append(val_metrics['Precision'])
        history['val_recall'].append(val_metrics['Recall'])
        history['val_f1'].append(val_metrics['F1'])
        
        # Log to TensorBoard
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Metrics/mIoU', val_metrics['mIoU'], epoch)
        writer.add_scalar('Metrics/Precision', val_metrics['Precision'], epoch)
        writer.add_scalar('Metrics/Recall', val_metrics['Recall'], epoch)
        writer.add_scalar('Metrics/F1', val_metrics['F1'], epoch)
        
        # Save best model
        if model_save_path and val_metrics['mIoU'] > best_miou:
            best_miou = val_metrics['mIoU']
            model.save(model_save_path)
            print(f"New best model saved with mIoU: {best_miou:.4f}")
    
    # Close TensorBoard writer
    writer.close()
    
    return history
